# Qwen2.5-1.5B-Instruct QLoRA Fine-Tune on nhankins/legal_contracts

This notebook fine-tunes a legal QA model using the CUAD-derived dataset on Hugging Face.
Target: `Qwen/Qwen2.5-1.5B-Instruct` on a T4 (16GB) with QLoRA.


In [ ]:
# Install dependencies
!pip -q install -U transformers datasets peft bitsandbytes accelerate

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Optional: Hugging Face token to avoid rate limits
import os
# os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig

model_name = 'Qwen/Qwen2.5-1.5B-Instruct'
output_dir = 'qwen2.5-1.5b-legal-qlora'
max_len = 1024  # T4-friendly
num_train_epochs = 1
train_limit = None  # set to int for quick runs (e.g., 2000)


In [ ]:
# Load CUAD SQuAD-style QA split
ds = load_dataset('nhankins/legal_contracts', 'squad_v1')
train_raw = ds['train']
eval_raw = ds['validation']

if train_limit:
    train_raw = train_raw.select(range(min(train_limit, len(train_raw))))

def format_example(example):
    question = example['question'].strip()
    context = example['context'].strip()
    answer = example['answers']['text'][0].strip() if example['answers']['text'] else 'Not found in provided context.'

    prompt = (
        'You are a legal contract assistant. Answer using only the provided context.\n\n'
        f'Question: {question}\n\n'
        f'Context:\n{context}\n\n'
        f'Answer:\n{answer}'
    )
    return {'text': prompt}

train = train_raw.map(format_example, remove_columns=train_raw.column_names)
eval_ = eval_raw.map(format_example, remove_columns=eval_raw.column_names)

In [ ]:
# Tokenize
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4'
)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=max_len)

train_tok = train.map(tokenize, batched=True, remove_columns=['text'])
eval_tok = eval_.map(tokenize, batched=True, remove_columns=['text'])

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora_config)

In [ ]:
# Train
args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=num_train_epochs,
    fp16=True,
    logging_steps=50,
    eval_steps=200,
    save_steps=200,
    evaluation_strategy='steps',
    save_total_limit=2,
    report_to='none'
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=data_collator,
)

trainer.train()
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

In [ ]:
# Quick sanity check
prompt = (
    'You are a legal contract assistant. Answer using only the provided context.\n\n'
    'Question: Who are the contract-holders?\n\n'
    'Context:\nThe contract-holder Name: Aya Ghaleb & Niccolo Forzano.\n\n'
    'Answer:'
)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(output[0], skip_special_tokens=True))